# Parte 2 — Tokenización y vocabularios
Entrada: `data/spa_normalized.tsv` (texto normalizado en el EDA, **sin** filtro de longitud).
Salida: tokenizador, filtro de longitud definitivo, splits train/val/test y vocabularios.

## 10. Elección del tokenizador
Decisión abierta del EDA: ¿`don't` / `Tom's` como una palabra (A) o separados en `don` + `'t` / `Tom` + `'s` (B)?

In [1]:
import os, re, csv, random, collections
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
PROJ = "/content/drive/MyDrive/PROYECTO TRADUCTOR"
DATA = os.path.join(PROJ, "data")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)

base = pd.read_csv(os.path.join(DATA, "spa_normalized.tsv"), sep="\t", header=None,
                   names=["english", "spanish", "attribution"],
                   quoting=csv.QUOTE_NONE, dtype=str, keep_default_na=False, encoding="utf-8")
print("Filas cargadas:", f"{len(base):,}", "(esperado: 144,215)")
display(base[["english", "spanish"]].sample(5, random_state=SEED))

Mounted at /content/drive
Filas cargadas: 144,215 (esperado: 144,215)


,english,spanish
31486,It's too far to walk.,Es demasiado lejos para ir andando.
93534,Come over here and give me a kiss.,Ven para acá y dame un beso.
74169,Tom's remark was insensitive.,La observación de Tom fue insensible.
106812,There's someone I'd like you to meet.,Hay alguien que me gustaría que conocierais.
71409,I have just finished my work.,Recién acabo de terminar mi trabajo.


In [2]:
# A: palabra (con apóstrofes internos pegados) | signo de puntuación suelto
TOK_A = re.compile(r"\w+(?:'\w+)*|[^\w\s]")
# B: palabra | sufijo de contracción inglesa separado ('s 't 'm 're 'll 've 'd) | signo suelto
TOK_B = re.compile(r"\w+|'(?:s|t|m|re|ll|ve|d)\b|[^\w\s]")

print("=== Ejemplos de tokenización ===")
tests = ["I don't know Tom's sister.", "How are you?", "¿Cómo estás, Tom?",
         '"Is he lying?" "Obviously."', "It's 2:30, o'clock.", "Trabajo para McDonald's."]
tests += list(base["english"].sample(3, random_state=SEED))
for s in tests:
    print(f"\n{s!r}\n  A: {TOK_A.findall(s)}\n  B: {TOK_B.findall(s)}")

def analyze(tok_re):
    en_t = base["english"].map(tok_re.findall)
    es_t = base["spanish"].map(tok_re.findall)
    out = {}
    for lang, toks in [("EN", en_t), ("ES", es_t)]:
        cnt = collections.Counter(t for ts in toks for t in ts)
        total = sum(cnt.values())
        out[f"{lang} tokens totales"] = total
        out[f"{lang} tipos distintos"] = len(cnt)
        out[f"{lang} tipos con freq=1"] = sum(1 for c in cnt.values() if c == 1)
        for mf in [2, 3, 5]:
            out[f"{lang} tipos con freq>={mf}"] = sum(1 for c in cnt.values() if c >= mf)
            out[f"{lang} % tokens que serían <UNK> (min_freq={mf})"] = sum(c for c in cnt.values() if c < mf) / total * 100
        lens = toks.map(len)
        out[f"{lang} media tokens/oración"] = lens.mean()
        out[f"{lang} P99 tokens"] = lens.quantile(0.99)
    both = np.maximum(en_t.map(len), es_t.map(len))
    for L in [25, 27, 30]:
        out[f"Pares con ambos lados <= {L} tokens (%)"] = (both <= L).mean() * 100
    return out

cmp_tok = pd.DataFrame({"A: contracciones pegadas": analyze(TOK_A),
                        "B: sufijos separados": analyze(TOK_B)})
print("\n=== Comparación A vs B (sobre todo el dataset; solo para decidir el diseño) ===")
display(cmp_tok.round(3))

=== Ejemplos de tokenización ===

"I don't know Tom's sister."
  A: ['I', "don't", 'know', "Tom's", 'sister', '.']
  B: ['I', 'don', "'t", 'know', 'Tom', "'s", 'sister', '.']

'How are you?'
  A: ['How', 'are', 'you', '?']
  B: ['How', 'are', 'you', '?']

'¿Cómo estás, Tom?'
  A: ['¿', 'Cómo', 'estás', ',', 'Tom', '?']
  B: ['¿', 'Cómo', 'estás', ',', 'Tom', '?']

'"Is he lying?" "Obviously."'
  A: ['"', 'Is', 'he', 'lying', '?', '"', '"', 'Obviously', '.', '"']
  B: ['"', 'Is', 'he', 'lying', '?', '"', '"', 'Obviously', '.', '"']

"It's 2:30, o'clock."
  A: ["It's", '2', ':', '30', ',', "o'clock", '.']
  B: ['It', "'s", '2', ':', '30', ',', 'o', "'", 'clock', '.']

"Trabajo para McDonald's."
  A: ['Trabajo', 'para', "McDonald's", '.']
  B: ['Trabajo', 'para', 'McDonald', "'s", '.']

"It's too far to walk."
  A: ["It's", 'too', 'far', 'to', 'walk', '.']
  B: ['It', "'s", 'too', 'far', 'to', 'walk', '.']

'Come over here and give me a kiss.'
  A: ['Come', 'over', 'here', 'and', 'give', 

,A: contracciones pegadas,B: sufijos separados
EN tokens totales,1052679.000,1099213.000
EN tipos distintos,16352.000,15935.000
EN tipos con freq=1,5603.000,5414.000
EN tipos con freq>=2,10749.000,10521.000
EN % tokens que serían <UNK> (min_freq=2),0.532,0.493
EN tipos con freq>=3,8459.000,8282.000
EN % tokens que serían <UNK> (min_freq=3),0.967,0.900
EN tipos con freq>=5,6326.000,6194.000
EN % tokens que serían <UNK> (min_freq=5),1.656,1.545
EN media tokens/oración,7.299,7.622


## 11. Tokenizador final, filtro de longitud y diagnóstico de grupos
Decisión de la sección 10: **tokenizador A** (`don't`, `Tom's` pegados; puntuación como token).
Filtro de longitud: `MAX_TOKENS = 25` de contenido (27 con `<SOS>`/`<EOS>`).
Antes del split, comprobamos cómo quedan los grupos para evitar leakage.

In [4]:
# =========================
# 11. TOKENIZADOR FINAL + FILTRO + DIAGNÓSTICO DE GRUPOS
# =========================
TOKEN_RE = TOK_A                       # decisión de la sección 10
def tokenize(text):
    return TOKEN_RE.findall(text)

MAX_TOKENS = 25                        # tokens de contenido (sin <SOS>/<EOS>)

data = base.copy()
data["en_tokens"] = data["english"].map(tokenize)
data["es_tokens"] = data["spanish"].map(tokenize)
max_len = np.maximum(data["en_tokens"].map(len), data["es_tokens"].map(len))
keep = max_len <= MAX_TOKENS
print(f"Pares antes del filtro : {len(data):,}")
print(f"Eliminados (> {MAX_TOKENS} tokens): {(~keep).sum():,}  (el EDA anticipó 109)")
data = data[keep].reset_index(drop=True)
print(f"Pares después          : {len(data):,}  (el EDA anticipó 144,106)")

# --- Grupos: componentes conexas del grafo inglés <-> español (claves normalizadas) ---
def norm_key(s):
    return re.sub(r"[^\w\s]", "", s.lower()).strip()

key_en = data["english"].map(norm_key)
key_es = data["spanish"].map(norm_key)

parent = {}
def find(x):
    while parent.setdefault(x, x) != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb

for e, s in zip(key_en, key_es):
    union(("en", e), ("es", s))

gid = {}
data["group"] = [gid.setdefault(find(("en", e)), len(gid)) for e in key_en]

sizes = data.groupby("group").size()
print("\n=== Diagnóstico de grupos ===")
print(f"Frases inglesas exactas distintas            : {data['english'].nunique():,}")
print(f"Claves inglesas normalizadas distintas       : {key_en.nunique():,}")
print(f"Grupos (componentes conexas EN<->ES)         : {len(sizes):,}")
print(f"Tamaño de grupo -> mediana: {sizes.median():.0f} | media: {sizes.mean():.2f} | "
      f"P99: {sizes.quantile(0.99):.0f} | máx: {sizes.max():,}")
print(f"Grupos de 1 fila: {(sizes == 1).sum():,} ({(sizes == 1).mean()*100:.1f}% de los grupos)")
print("\nLos 8 grupos más grandes (nº de filas):", sizes.sort_values(ascending=False).head(8).tolist())
print(f"Filas en el grupo más grande: {sizes.max():,} ({sizes.max()/len(data)*100:.3f}% del dataset)")

print("\nEjemplos del grupo más grande:")
top = sizes.idxmax()
display(data[data["group"] == top][["english", "spanish"]].sample(min(10, sizes.max()), random_state=SEED))

Pares antes del filtro : 144,215
Eliminados (> 25 tokens): 109  (el EDA anticipó 109)
Pares después          : 144,106  (el EDA anticipó 144,106)

=== Diagnóstico de grupos ===
Frases inglesas exactas distintas            : 121,385
Claves inglesas normalizadas distintas       : 121,222
Grupos (componentes conexas EN<->ES)         : 112,998
Tamaño de grupo -> mediana: 1 | media: 1.28 | P99: 5 | máx: 68
Grupos de 1 fila: 93,843 (83.0% de los grupos)

Los 8 grupos más grandes (nº de filas): [68, 58, 43, 43, 42, 39, 39, 36]
Filas en el grupo más grande: 68 (0.047% del dataset)

Ejemplos del grupo más grande:


,english,spanish
33850,You can put it there.,Podéis ponerla ahí.
33820,You can put it there.,Lo puedes dejar ahí.
33808,You can put it there.,Lo podéis poner ahí.
33813,You can put it there.,Lo puede poner allí.
33832,You can put it there.,Lo pueden dejar ahí.
33845,You can put it there.,Puedes ponerla allí.
33862,You can put it there.,Vosotras la podéis poner ahí.
33809,You can put it there.,Lo podéis poner allí.
33865,You can put it there.,Ustedes lo pueden poner ahí.
33816,You can put it there.,Lo pueden poner ahí.


## 12. Split train / validation / test (por grupos, sin leakage)
Proporción 90/5/5 medida en **filas**, asignando **grupos completos** a cada conjunto.
Verificaciones: ningún grupo, inglés ni español aparece en más de un conjunto, y los conjuntos tienen distribuciones parecidas.

In [5]:
# =========================
# 12. SPLIT POR GRUPOS
# =========================
import json
TRAIN_FRAC, VAL_FRAC = 0.90, 0.05          # test = el resto
rng = np.random.default_rng(SEED)

perm = rng.permutation(sizes.index.to_numpy())               # orden aleatorio de grupos
cum = np.cumsum(sizes.loc[perm].to_numpy()) / len(data)      # fracción acumulada de FILAS
split_of_group = np.where(cum <= TRAIN_FRAC, "train",
                  np.where(cum <= TRAIN_FRAC + VAL_FRAC, "val", "test"))
data["split"] = data["group"].map(dict(zip(perm, split_of_group)))
data["key_en"], data["key_es"] = key_en, key_es
data["en_len"] = data["en_tokens"].map(len)
data["es_len"] = data["es_tokens"].map(len)

# --- Tamaños ---
print("=== Tamaños ===")
summary = data.groupby("split").agg(
    filas=("english", "size"),
    grupos=("group", "nunique"),
    ingles_unicos=("english", "nunique"),
    media_en_toks=("en_len", "mean"),
    media_es_toks=("es_len", "mean"),
    p95_en=("en_len", lambda s: s.quantile(0.95)),
    p95_es=("es_len", lambda s: s.quantile(0.95)),
).loc[["train", "val", "test"]]
summary["% filas"] = summary["filas"] / len(data) * 100
display(summary.round(3))

# --- Verificaciones anti-leakage ---
print("\n=== Verificaciones anti-leakage (todo debe ser 0) ===")
print("Grupos presentes en más de un split:", int((data.groupby("group")["split"].nunique() > 1).sum()))
for col, name in [("english", "inglés exacto"), ("spanish", "español exacto"),
                  ("key_en", "inglés normalizado"), ("key_es", "español normalizado")]:
    sets = {sp: set(data.loc[data["split"] == sp, col]) for sp in ["train", "val", "test"]}
    print(f"{name:20s} train∩val={len(sets['train'] & sets['val'])} | "
          f"train∩test={len(sets['train'] & sets['test'])} | val∩test={len(sets['val'] & sets['test'])}")

# --- ¿Los splits son comparables? (traducciones múltiples por grupo) ---
gsize = data["group"].map(sizes)
print("\n=== % de filas que pertenecen a grupos con 2+ filas, por split ===")
print((data.assign(multi=gsize >= 2).groupby("split")["multi"].mean().loc[["train", "val", "test"]] * 100).round(2).to_string())

# --- Guardar splits en Drive ---
SPLIT_DIR = os.path.join(DATA, "splits")
os.makedirs(SPLIT_DIR, exist_ok=True)
for sp in ["train", "val", "test"]:
    d = data[data["split"] == sp]
    with open(os.path.join(SPLIT_DIR, f"{sp}.tsv"), "w", encoding="utf-8", newline="\n") as f:
        for e, s, a, g in zip(d["english"], d["spanish"], d["attribution"], d["group"]):
            f.write(f"{e}\t{s}\t{a}\t{g}\n")
with open(os.path.join(SPLIT_DIR, "split_log.json"), "w", encoding="utf-8") as f:
    json.dump({"seed": SEED, "fracs": {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": round(1 - TRAIN_FRAC - VAL_FRAC, 4)},
               "max_tokens": MAX_TOKENS, "filas": {sp: int((data["split"] == sp).sum()) for sp in ["train", "val", "test"]},
               "grupos_total": int(len(sizes))}, f, indent=2)
print("\nGuardado en:", SPLIT_DIR, "->", os.listdir(SPLIT_DIR))

=== Tamaños ===


,filas,grupos,ingles_unicos,media_en_toks,media_es_toks,p95_en,p95_es,% filas
split,,,,,,,,
train,129695,101809,109360,7.292,7.191,12.0,12.0,90.0
val,7205,5500,5950,7.180,7.085,12.0,12.0,5.0
test,7206,5689,6075,7.233,7.159,12.0,12.0,5.0



=== Verificaciones anti-leakage (todo debe ser 0) ===
Grupos presentes en más de un split: 0
inglés exacto        train∩val=0 | train∩test=0 | val∩test=0
español exacto       train∩val=0 | train∩test=0 | val∩test=0
inglés normalizado   train∩val=0 | train∩test=0 | val∩test=0
español normalizado  train∩val=0 | train∩test=0 | val∩test=0

=== % de filas que pertenecen a grupos con 2+ filas, por split ===
split
train    34.82
val      36.95
test     33.87

Guardado en: /content/drive/MyDrive/PROYECTO TRADUCTOR/data/splits -> ['train.tsv', 'val.tsv', 'test.tsv', 'split_log.json']


## 13. Vocabularios (construidos solo con train)
Tokens especiales: `<PAD>=0`, `<UNK>=1`, `<SOS>=2`, `<EOS>=3`.
`min_freq` se elige con la tasa de `<UNK>` en **validación** (nunca en test).

In [6]:
# =========================
# 13. VOCABULARIOS (solo train) + DIAGNÓSTICO DE <UNK> EN VALIDACIÓN
# =========================
PAD, UNK, SOS, EOS = "<PAD>", "<UNK>", "<SOS>", "<EOS>"
SPECIALS = [PAD, UNK, SOS, EOS]           # índices 0, 1, 2, 3

train = data[data["split"] == "train"]
val   = data[data["split"] == "val"]

def count_tokens(token_lists):
    c = collections.Counter()
    for ts in token_lists:
        c.update(ts)
    return c

cnt_en, cnt_es = count_tokens(train["en_tokens"]), count_tokens(train["es_tokens"])

def build_vocab(counter, min_freq):
    words = [w for w, c in sorted(counter.items(), key=lambda kv: (-kv[1], kv[0])) if c >= min_freq]
    itos = SPECIALS + words                # index -> palabra
    stoi = {w: i for i, w in enumerate(itos)}   # palabra -> index
    return stoi, itos

def unk_stats(token_lists, stoi):
    total = unk = sents_with_unk = 0
    for ts in token_lists:
        u = sum(1 for t in ts if t not in stoi)
        total += len(ts); unk += u; sents_with_unk += (u > 0)
    return unk / total * 100, sents_with_unk / len(token_lists) * 100

# --- 13.1 ¿Qué min_freq conviene? (medido en VALIDACIÓN) ---
rows = []
for mf in [1, 2, 3, 5]:
    stoi_en, _ = build_vocab(cnt_en, mf)
    stoi_es, _ = build_vocab(cnt_es, mf)
    en_tok, en_sent = unk_stats(val["en_tokens"], stoi_en)
    es_tok, es_sent = unk_stats(val["es_tokens"], stoi_es)
    rows.append({"min_freq": mf, "vocab EN": len(stoi_en), "vocab ES": len(stoi_es),
                 "val EN %tokens UNK": en_tok, "val EN %oraciones c/UNK": en_sent,
                 "val ES %tokens UNK": es_tok, "val ES %oraciones c/UNK": es_sent})
print("=== 13.1 min_freq vs tasa de <UNK> en validación ===")
display(pd.DataFrame(rows).round(3))

# --- 13.2 Vocabulario con el min_freq candidato ---
MIN_FREQ = 2
stoi_en, itos_en = build_vocab(cnt_en, MIN_FREQ)
stoi_es, itos_es = build_vocab(cnt_es, MIN_FREQ)
print(f"\n=== 13.2 Vocabularios con min_freq={MIN_FREQ} ===")
print(f"EN: {len(itos_en):,} entradas | ES: {len(itos_es):,} entradas")
print("Primeras entradas EN:", itos_en[:14])
print("Primeras entradas ES:", itos_es[:14])
for sp in [PAD, UNK, SOS, EOS]:
    assert stoi_en[sp] == stoi_es[sp] == SPECIALS.index(sp)

# --- 13.3 ¿Qué palabras son <UNK> en validación? ---
def unk_words(token_lists, stoi):
    return collections.Counter(t for ts in token_lists for t in ts if t not in stoi)

def categorize(t):
    if any(ch.isdigit() for ch in t): return "contiene dígitos"
    if t[0].isupper(): return "capitalizada (nombre propio / inicio)"
    return "minúscula (palabra rara)"

for lang, col, stoi in [("EN", "en_tokens", stoi_en), ("ES", "es_tokens", stoi_es)]:
    uw = unk_words(val[col], stoi)
    total = sum(uw.values())
    cats = collections.Counter()
    for w, c in uw.items():
        cats[categorize(w)] += c
    print(f"\n[{lang}] {total:,} tokens <UNK> en validación ({len(uw):,} palabras distintas)")
    print("  por categoría:", {k: f"{v/total*100:.1f}%" for k, v in cats.most_common()})
    print("  top 25:", uw.most_common(25))

# --- 13.4 Ejemplos con <UNK> ---
print("\n=== 13.4 Ejemplos de validación con <UNK> ===")
mask = val["en_tokens"].map(lambda ts: any(t not in stoi_en for t in ts))
for _, r in val[mask].sample(6, random_state=SEED).iterrows():
    print("EN:", " ".join(t if t in stoi_en else "<UNK>" for t in r["en_tokens"]))
    print("ES:", " ".join(t if t in stoi_es else "<UNK>" for t in r["es_tokens"]), "\n")

=== 13.1 min_freq vs tasa de <UNK> en validación ===


,min_freq,vocab EN,vocab ES,val EN %tokens UNK,val EN %oraciones c/UNK,val ES %tokens UNK,val ES %oraciones c/UNK
0,1,15654,30263,0.686,4.552,1.554,10.257
1,2,10231,17512,1.125,7.231,2.468,15.975
2,3,8037,12852,1.521,9.521,3.213,20.194
3,5,6009,8795,2.138,12.908,4.469,27.217



=== 13.2 Vocabularios con min_freq=2 ===
EN: 10,231 entradas | ES: 17,512 entradas
Primeras entradas EN: ['<PAD>', '<UNK>', '<SOS>', '<EOS>', '.', 'I', 'to', 'the', 'Tom', 'you', 'a', '?', 'is', 'in']
Primeras entradas ES: ['<PAD>', '<UNK>', '<SOS>', '<EOS>', '.', 'de', 'que', 'Tom', 'a', '?', '¿', 'la', 'en', 'el']

[EN] 582 tokens <UNK> en validación (508 palabras distintas)
  por categoría: {'minúscula (palabra rara)': '77.7%', 'capitalizada (nombre propio / inicio)': '21.5%', 'contiene dígitos': '0.9%'}
  top 25: [('Smell', 5), ('dice', 4), ('robots', 4), ('conform', 4), ('Fantastic', 3), ('brightened', 3), ('pusher', 3), ('repeats', 3), ('unclear', 3), ('Terrific', 2), ('pushing', 2), ('outrank', 2), ('doomed', 2), ('booked', 2), ('squabbling', 2), ('geek', 2), ('Checks', 2), ('resilient', 2), ('partied', 2), ("Now's", 2), ('Elves', 2), ('pointy', 2), ('consultant', 2), ('dentists', 2), ('cellar', 2)]

[ES] 1,260 tokens <UNK> en validación (1,123 palabras distintas)
  por categor